# MIRI MRS calibration with a custom fringe flat

This notebook runs the MIRI MRS reduction with BREADS, replacing the JWST pipeline fringe correction by a fringe flat fitted directly on the stage 1 rate files:

1. **Stage 1**: `uncal` → `rate` files.
2. **Fringe flat**: Fabry-Perot fringe model fitted column by column on the rate files (`get_flat_parallelized`), then divided out of the rate files (`flat_fringing_stage1`).
3. **Stage 2**: `rate` → `cal` files, with the pipeline `fringe`, `residual_fringe`, `flat_field` and `straylight` steps skipped.
4. **Stage 3** (optional): cube building.

## Environment

The thread limits must be set **before** numpy is imported (first cell of the notebook): the fringe fit runs one process per CPU, and multi-threaded BLAS in each of them would oversubscribe the machine.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache")
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

In [ ]:
from breads.jwst_reduction.reduction_utils import run_stage1_miri, run_stage2_miri, run_stage3_miri, flat_fringing_stage1
from breads.jwst_tools.function_fitting_fringes import get_flat_parallelized

## Configuration

Edit the paths below. `uncaldir` contains the `*_uncal.fits` files; the reduced products are written in `uncaldir/<target name>/<band>/`. The fitted fringe flats are written in `FLAT_PATH/<band>/`.

In [ ]:
# Main directory for the data and reduced products (change those before running this block).

uncaldir = "/path/to/uncal/" #uncaldir is the location of all the uncal files for the target, the reference star if you want to use it later for fringe calibration and optional background observation
flat_dir = "/path/to/fringe_flats/" #Flat directory to save the flat fits file
os.environ['FLAT_PATH'] = flat_dir #Set the flat directory to the environment

list_bands = ['34B'] #subset of the MRS sub band you want to work on

In [ ]:
rate_files, targnames = run_stage1_miri(uncaldir, list_bands=list_bands, overwrite=False, maximum_cores="1", skip_dark=False, verbose=False)
#run_bkg_subtraction(uncaldir, targnames, list_bands=list_bands, overwrite=False)


science_targname = targnames[0] #We suppose here that the science targname is the first of the list. Check if this is the case for your data before running the other steps below!


print(targnames)
print(science_targname)

## Stage 1

`run_stage1_miri` is a wrapper around the jwst pipeline and returns the rate files and the target names found in the headers.

Some useful optional arguments:
- "list_bands": MRS has 6 files for 12 bands. Each detector images 2 sub bands simultaneously. You can select which subset of sub bands you want to reduce here. For instance '12A', '12B', '12C', '34A', '34B' or '34C'
- "skip_dark": The MIRI dark current can be chaotic, sometimes a background subtraction does a better job has it is often taken closer in time from the science exposure. You can set this to True if you want to skip the dark and replace it later with background subtraction (use run_bkg_subtraction).

## Fringe flat
There are two ways to calibrate the fringes.

The first method calls the function `run_miri_flat_running_mean` is an empirical correction of the fringes. It makes the assumption that the fringes pattern remains the same for a reference star observed with the same star pointing and dithering. The method uses a gaussian filtering per column to retrieve the fringes oscillations around a continuum. This is based on _Gasman et al. 2023_ technique.

The second method is by using `get_flat_parallelized` that fits the Fabry-Perot fringe model on every detector column (one column per task, all CPUs by default; `n_procs=1` runs sequentially for debugging). With `N_continuum`, `N_D` and `N_finesse` set to `None`, the number of spline nodes is chosen per band. `skip_4=True` skips channel 4 which has a reduced throughput and is often skipped for high-contrast.

Note that it assumes that the
`flat_fringing_stage1` then divides each rate file by its best-matching flat and writes the result in `stage1_flat/`.

In [ ]:
get_flat_parallelized(uncaldir, science_targname, list_bands=list_bands, bkg_sub=False, snr_thresh=-1e5,
                      spectrum=None, N_continuum=None, N_D=None, N_finesse=None,
                      mask_star_lines=True, fast=True, skip_4=True, overwrite=True)

In [ ]:
flat_fringing_stage1(uncaldir, science_targname, list_bands=list_bands, flat_path=None, flat_extended=True, bkg_sub=False, overwrite=True)

## Stage 2

The custom-flatted rate files (`custom_flatted=True`) are processed with the pipeline fringe, residual fringe, flat field and straylight steps skipped.

Some useful optional arguments:
-custom_flatted: Set to True if you used one of the custom fringes correction mentioned just above.
-custom_bkg_sub: Set to True if you used the background subtraction.

In [ ]:
cal_files = run_stage2_miri(uncaldir, targnames[0], list_bands=list_bands, custom_flatted=True, custom_bkg_sub=False,
                            skip_cubes=True, skip_fringe=True, skip_residual_fringes=True,
                            skip_flatfield=True, skip_straylight=True, overwrite=True, verbose=False)

## Stage 3 (optional)
The stage 3 is producing the drizzle cube. We do not compute the CCF with them with breads but it can be helpful to visualize the PSF cube before subtraction.

In [ ]:
run_stage3 = False
if run_stage3:
    run_stage3_miri(uncaldir, targnames[0], list_bands=list_bands, overwrite=False, verbose=False)